# CIRCA Interactive Training Environment

This notebook allows you to run your thesis experiment phases interactively. 
Because your `train_engine.py` script contains crucial project-specific logic (like CLAHE+Gamma preprocessing, W&B logging rules, low-VRAM stability patches, and OpenVINO INT8 exports), the best way to train here is by calling the script directly using Jupyter's shell (`!`) commands.

In [ ]:
# 0. Setup and Verification
import torch
from ultralytics import YOLO

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Phase 1 — Vanilla Baseline (Ablation Control)
Trains YOLOv12-S on raw images without the CIRCA preprocessing pipeline. Used as the baseline to quantify the lift from your enhancements.

In [ ]:
!python train_engine.py --mode train --variant s --id 001 --desc Baseline_Vanilla --epochs 50 --clear

## Phase 2 — CIRCA-Aligned Baseline
Trains YOLOv12-S on the **preprocessed** dataset (CLAHE + Gamma). This confirms the preprocessing pipeline works and provides a realistic target for the hyperparameter tuner to beat.

In [ ]:
!python train_engine.py --mode train --variant s --id 002 --desc Baseline_CIRCA --epochs 100 --preproc

## Phase 3 — Hyperparameter Optimisation (HPO)
Executes the genetic tuning algorithm over 150 iterations. 
**Warning:** This cell will take a very long time to complete.

In [ ]:
!python train_engine.py --mode tune --variant s --id 003 --desc HPO --epochs 30 --iterations 150 --preproc

## Phase 4 — Final Model Trainings (using HPO Config)
Once Phase 3 generates `best_hyperparameters.yaml`, you can train the N, S, and M variants.

In [ ]:
# Example: Train YOLOv12-N with tuned config
# !python train_engine.py --mode train --variant n --id 004 --desc Final_HPO --epochs 200 --preproc --cfg runs/detect/CIRCA_V12S_003_TUNE_HPO/best_hyperparameters.yaml